
Geo Data Science with Python,
Prof. Susanna Werth, VT Geosciences

# Learning Models


This notebook is accompanied by the lecture L06 presentation slides.

Note: This notebook was updated after the lecture on Monday, October 13.

---


Content:
-------
- **A.** Simple Linear Regression, Optimization & Evaluation
- **B.** Multiple Linear Regression
- **C.** Cross-Validation
- **D.** Polynomial Models
 


Preparation & Data Retrieval:
----------------------------



In [ ]:

# On Google Colab, install the following packages...

# pip install netCDF4


In [ ]:

# Import standard packages
import requests
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset   # on google colab you have to install this
import pandas as pd
import seaborn as sns

# let's set a standard size for figures (a bit smaller, so they fit on the screen)
from matplotlib import rcParams
rcParams['figure.figsize'] = [4, 3]
rcParams.update({'font.size': 12})

# note: more new packages, will be imported below


In [ ]:

# Download Data
raw_url = "https://raw.githubusercontent.com/GeoPythonVT/geosf25_material/main/data_downloadArchive/GLDAS_CLSM10_M.A202506.021.nc4"
out = "./dataLoaded/GLDAS_CLSM10_M.A202506.021.nc4"
with requests.get(raw_url, stream=True) as r:
    r.raise_for_status()
    with open(out, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024): # use this download version for large files
            if chunk:
                f.write(chunk)

gldasDat = Dataset('./dataLoaded/GLDAS_CLSM10_M.A202506.021.nc4')
print([ e for e in gldasDat.dimensions ]) # lists all dimensions in the dataset
print([ e for e in gldasDat.variables ]) # lists all variables in the dataset (here muted)


In [ ]:

# Read data variables as masked arrays from file
ilatMin = 72    # equator is near index 60
ilatMax = 149   # max index is 149
ilonMin = 10     # Greenwich is near index 180
ilonMax = 160   # max index is 359

lat = gldasDat.variables['lat'][ilatMin:ilatMax] # data shape is shape is lat x lon
lon = gldasDat.variables['lon'][ilonMin:ilonMax]

tair = gldasDat.variables['Tair_f_inst'][0, ilatMin:ilatMax,ilonMin:ilonMax]-272.15         # Temperature, Celcius
surfT = gldasDat.variables['AvgSurfT_inst'][0, ilatMin:ilatMax,ilonMin:ilonMax]-272.15      # Average Surface Skin temperature, Celcius
rainf = gldasDat.variables['Rainf_tavg'][0, ilatMin:ilatMax,ilonMin:ilonMax]*60*60*24*30/10 # Rain precipitation rate, cm/month
snowf = gldasDat.variables['Snowf_tavg'][0, ilatMin:ilatMax,ilonMin:ilonMax]*60*60*24*30/10 # Snow precipitation rate, cm/month
qs = gldasDat.variables['Qs_acc'][0, ilatMin:ilatMax,ilonMin:ilonMax]/10                    # Storm surface runoff, cm w.eq.
evap = gldasDat.variables['Evap_tavg'][0, ilatMin:ilatMax,ilonMin:ilonMax]*60*60*24*30/10   # Evapotranspiration, cm/month
soilmS = gldasDat.variables['SoilMoist_P_inst'][0, ilatMin:ilatMax,ilonMin:ilonMax]/10      # Profile Soil moisture, cm w.eq.
snowS = gldasDat.variables['SWE_inst'][0, ilatMin:ilatMax,ilonMin:ilonMax]/10               # Snow depth water equivalent, cm w.eq.
tws = gldasDat.variables['TWS_inst'][0, ilatMin:ilatMax,ilonMin:ilonMax]/10                 # Terrestrial water storage, cm w.eq.


In [ ]:
# let's see which part of the map we picked
fig, axes = plt.subplots(1, 2, figsize=(8, 2))
im1 = axes[0].pcolormesh(lon, lat, surfT)  # rainf*60*60*24*30/10
axes[0].set_title("Rainfall")
im2 = axes[1].pcolormesh(lon, lat, tws/10)
axes[1].set_title("TWS")
cbar = fig.colorbar(im1, ax=axes, orientation='vertical', fraction=0.046, pad=0.1)
cbar.set_label("cm/month")  
cbar = fig.colorbar(im2, ax=axes, orientation='vertical', fraction=0.046, pad=0.1)
cbar.set_label("cm")  
#plt.tight_layout()
plt.show()

In [ ]:

# Fill noData with nan and store in Pandas Dataframe
A = np.array([ tair.filled(np.nan).ravel(),   # Degree Celcius
               surfT.filled(np.nan).ravel(),  # Degree Celcius
               rainf.filled(np.nan).ravel(),  # cm per month
               snowf.filled(np.nan).ravel(),  # cm per month
               qs.filled(np.nan).ravel(),     # cm w.eq.
               evap.filled(np.nan).ravel(),   # cm per month
               soilmS.filled(np.nan).ravel(), # cm w.eq.
               snowS.filled(np.nan).ravel(),  # cm w.eq.
               tws.filled(np.nan).ravel() ])  # cm w.eq.
AcolNames = [ "tair", "surfT", "rainf", "snowf", "qs", "evap", "soilmS", "snowS", "tws"]   # name the columns
df = pd.DataFrame(A.T, columns=AcolNames)


---
# A. Simple Linear Regression, Optimization & Evaluation


### Correlations in the data

In [ ]:

# Pairplot (visually assess linearity of correlations) of selected variables
sns.pairplot(
    data = df[["tair", "surfT", "rainf", "qs", "soilmS", "snowS", "tws"]], 
    kind = 'scatter', height = 1.2, corner = True,
    diag_kind='kde',
    plot_kws = {'color':'#CF4420', 'edgecolor':'black', 'alpha':0.1},
    diag_kws = {'color': '#630031'}
    )


**Task:** Discuss the relationships in between variables. How would you describe them? Which relationships seem random, strongly linear, non-linear?


**Task:**
Formulate a regression model (on paper) to predict surface temperatur from air temperature.


In the code cell below, uncomment the sns.regplot() command, to see the output of seaborn regression feature.

In [ ]:
fig = plt.figure()
sns.scatterplot(data=df, x="tair", y="surfT")  
#sns.regplot(data=df, x="tair", y="surfT", scatter=False, color="orange")
plt.show()

### Simple Linear Regression / Line Fitting using `sklearn`


In [ ]:

# convert masked arrays into plain arrays, removing noData values
OceanMask = tair.mask | surfT.mask
x = tair.data[~OceanMask]  # indipendent variable
y = surfT.data[~OceanMask]      # dependent variable


In [ ]:
from sklearn.linear_model import LinearRegression

# Reshape x to 2D for sklearn
X = x.reshape(-1, 1)

model = LinearRegression()  # fits y = b0 + b1*x
model.fit(X, y)

# Predict y from x
y_est = model.predict(X)

# Print results
print("Slope (coef):", model.coef_[0])       # b1
print("Intercept (bias):", model.intercept_) # b0 automatically added to the model
print("R² score:", model.score(X, y))        # R2 Coefficient of determination


In [ ]:

# Plot model output
fig = plt.figure()
sns.scatterplot(data=df, x="tair", y="surfT")  
plt.plot(x, y_est,"k")
plt.xlabel("Tair"); plt.ylabel("Surface Temp")
plt.show()


### Solution via Inversion of a System of Linear Equations

Note that the np.polyfit functions is based on a L2 norm (least squares), that is solved via a pseudoe inverse of the design matrix. We could also reproduce this by writing our matrices as a linear equation system and then solving that accordingly.

In [ ]:
# Formulates system of equations (containing indipendent variables and intercept / bias of the regression)
A = np.column_stack((x, np.ones(x.shape)))   

# Estimates pseudo inverse of non-square A matrix
pEst = np.linalg.pinv(A)@y      

# Evaluates the model
y_est2 = A@pEst          

In [ ]:
pEst

Compare with above, the parameter results are basically the same, same for the plot.

In [ ]:

# Plot model output
fig = plt.figure()
sns.scatterplot(data=df, x="tair", y="surfT")  
plt.plot(x, y_est2,"k")
plt.xlabel("Tair"); plt.ylabel("Surface Temp")
plt.show()


### Using a different objective/cost function with `scipy`

Scipy's optimize function allows us to set our own cost function. Let's see how we can do that.
First, we define a few cost functions, to illustrate this.

In [ ]:
# Model definitions for different L-norms

# L_1-norm: Robust to outliers, yields simpler models
def L1norm(p0,A,l): 
    return np.sum(np.abs(A@p0 - l))

# L_2-norm: smooth function, penalizes large error, stable, unique solutions
def L2norm(p0,A,l):
    return np.sum(np.power(np.abs(A@p0 - l),2))

# L_inf-norm: Guarantees stability under worst-case error
def Linfnorm(p0,A,l):  
    return np.max(np.abs(A@p0 - l))


Calculating the optimization problem: estimating (inverting) the parameter. We solve this with using the `scipy.optimize` function `minimize`, which receives:
- a function, that defines the minimization problem (and is used to evaluate the model output),
- an initial guess for the model parameter (`p0`), and 
- the arguments of the model system (`A` and `y`), with `A` being the values of the desing matrix and `y` being the observations (true values).

Standard solver is a gradient-based approach ('BFGS').

In [ ]:
import scipy.optimize

# convert masked arrays into plain arrays, removing noData values
OceanMask = tair.mask | surfT.mask
x = tair.data[~OceanMask]  # indipendent variable
y = surfT.data[~OceanMask] # dependent variable

A = np.column_stack((x, np.ones_like(x)))  # fits whatever is defined by the design matrix A
# note: intercept has to be added in the Design matrix

p0 = np.array([0,0])
sol1 = scipy.optimize.minimize(L1norm,p0,args=(A,y), method='BFGS') # linear regression with L1-Norm, gradient based approach is standard
sol2 = scipy.optimize.minimize(L2norm,p0,args=(A,y))    # linear regression with L2-Norm
sol3 = scipy.optimize.minimize(Linfnorm,p0,args=(A,y))   # linear regression with Linf-Norm
sol1.nit  # number of iterations used

In [ ]:

# Estimating linear model output (lm) for set of input points (a)
lm1 = A @ sol1.x
lm2 = A @ sol2.x
lm3 = A @ sol3.x


In [ ]:
# Plotting results for linear models for all three norms
plt.figure()
plt.plot(x,y,'x',color='r')
plt.plot(x,lm1,'-',color='k',linewidth=1,label="$l_1$")    # L1
plt.plot(x,lm2,'-',color='g',linewidth=1,label='$l_2$')     # L2
plt.plot(x,lm3,'-',color='b',linewidth=1,label='$l_\infty$')# Linf
plt.xlabel('x'), plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
# For exercise A.1, to better visualize the impact of outliers,
# reduce the dataset as follows before optimization:

# Reduce dataset to emphasize on effect of a single outlier 
percentage = 0.01
arr = np.arange(len(x))  # example array
idx = np.random.choice(len(arr), size=int(len(arr)*percentage), replace=False)
x=x[idx] 
y=y[idx]

# Add single outlier (change to your liking)
x[35]=30  
y[35]=150 



--- 
## Exercise A.

1. Recalculate the last regression using `scipy.optimize.minimize` after inserting an outlier into the dataset. Compare the different results visually.
2. Do you get a different outcome for the different norms with and without outlier? How do outliers in the data affect the least-square optimization?

**Extra Credit**

3. Repeat the comparison for any other interesting pair of variables using simple linear regression.



---
# B. Multiple Linear Regresssion

Now we will use sklearn, because it provides a great function for multiple linear regression. Let's first revisit the simple regression with this function, then build a more complex model using multiple parameter.

### Multiple / General Regression Model

In [ ]:

# convert masked arrays into plain arrays, removing noData values
OceanMask = surfT.mask | rainf.mask | tair.mask
x1 = tair.data[~OceanMask]  # indipendent variable
x2 = rainf.data[~OceanMask]  # indipendent variable
y = surfT.data[~OceanMask]      # dependent variable


In [ ]:
from sklearn.linear_model import LinearRegression

# Stack columns into a big data matrix X
X = np.column_stack((x1, x2)) # add more variables here to increase the number of variables in the model

model = LinearRegression()    # fits y = b0 + b1*x + ... + bn*x
model.fit(X, y)

# Predict y from x
y_estMR = model.predict(X)

# Print results
print("Slope (coef):", model.coef_)          # b1 ... bn
print("Intercept (bias):", model.intercept_) # b0
print("R² score:", model.score(X, y))


### Goodness of fit criteria

In [ ]:

# Goodness of fit criteria
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy import stats

# Note: 
# - y is the observation of a physical variable
# - y_estMR is the model output for that same physical variable

# Correlation
r, p = stats.pearsonr(y, y_estMR)
rho, p = stats.spearmanr(y, y_estMR)

# Standard criteria using built-in functions of sklearn and scipy
R2  = r2_score(y, y_estMR)    # estimated using scipy.stats
RMSE = np.sqrt(mean_squared_error(y, y_estMR)) # estimated using sklearn.metrics 
MAE  = mean_absolute_error(y, y_estMR)

# AIC formula, likelihood based, manual calculation
n = len(y)
k = X.shape[1] + 1  # +1 for intercept
rss = np.sum((y - y_estMR)**2)
AIC = n * np.log(rss / n) + 2 * k

print(r, rho)
print(R2, RMSE, MAE, AIC)


--- 
## Exercise B.

1. Estimate a simple linear regression for predicting total water storage (`tws`) from soil moisture (`soilmS`) using `sklearn.linear_model.LinearRegression()`.
2. Now, successively add variables and see if they increase the accuracy of the model.
3. Visualize the outcome by plotting the change in goodness of fit measures over number of included parameters. Include the GoF measures correlation, R2 and AIC.
4. Discuss how you interpret the outcome.

**Extra Credit**

4. Find out if there are any variables that further increase the accuracy of the multiple linear regression for predicting surface temperature from air temperature and rainfall? Discuss the meaning.



---
# C. Cross-Validation

**Task:** How can we do a simple cross validation for the model of surface temperature predicted via air temperature?


In [ ]:
# convert masked arrays into plain arrays, removing noData values
OceanMask = qs.mask | snowS.mask
x = snowS.data[~OceanMask]  # indipendent variable
y = qs.data[~OceanMask]      # dependent variable

# Generate a subset of the original dataset, getting 90% of the data for training, 10% for testing
percentage = 0.9
arr = np.arange(len(x))  # example array
idx = np.random.choice(len(arr), size=int(len(arr)*percentage), replace=False)
all_idx = np.arange(len(x))
other_idx = np.setdiff1d(all_idx, idx)

# Training data
x_sub = x[idx] # selects randomly half of the values
y_sub = y[idx] 

# Testing data
x_sub2 = x[other_idx] # selects randomly half of the values
y_sub2 = y[other_idx] 

len(x_sub), len(x_sub2)


In [ ]:
# Reshape x to 2D for sklearn
X_sub = x_sub.reshape(-1, 1)   # training data
X_sub2 = x_sub2.reshape(-1, 1) # testing data

model = LinearRegression()  # fits y = b0 + b1*x
model.fit(X_sub, y_sub)

# Predict y from x
y_est_sub = model.predict(X_sub)
y_est_cval = model.predict(X_sub2)

# Print results
print("R² score:", model.score(X_sub, y_sub))     # Training score
print("R² score:", model.score(X_sub2, y_sub2))   # Testing score


--- 
## Exercise C.

1. Write a loop over the cross validation, that repeats the cross-validation 50 times and plots the repeat it at least 50 times.
2. Plot a boxplot for the resulting goodness of fit measure result, e.g., R2 values, from all cross-validation runs.

**Extra Credit**

3. Are there any variables that further increase the accuracy of the multiple linear regression for predicting surface temperature from air temperature and rainfall?



---
# D. Polynomial Models & Regularization

### Polynomial Models

In [ ]:

# convert masked arrays into plain arrays, removing noData values
OceanMask = tair.mask | surfT.mask
x = tair.data[~OceanMask]  # indipendent variable
y = surfT.data[~OceanMask] # dependent variable

# Estimate polynomial model using NumPy Polyfit uses least squares (L2norm), and solves via SVD
[a,b,c] = np.polyfit(x, y, 2) # returns polynomial coefficients, highest power first
# Note although not explicitly formualted in the code, the polynomial model is linearized as: y = p1 * x^2 + p2 * x + p3
# The function allows a quick fit of polynomials, without explicit formulation of a Design matrix. 
# Estimate model output
y_reg = a*x*x+b*x+c 

# Plot model output
fig = plt.figure()
sns.scatterplot(data=df, x="tair", y="surfT")  
plt.plot(x, y_reg,"k")
plt.xlabel("Tair"); plt.ylabel("Surface Temp")
plt.show()


--- 
## Exercise D. 

1. Use the function polyfit to fit a polynomial function, including three degrees of power over the predictors. Does the fit improve compared to the model of one or two degrees? Use andy goodness of fit criteria that bases on residuals (i.e., is not the correlation function).

**Extra credit**

2. You can also use the `scipy.optimize.minimize()` function to solve the polynomial model. How many columns must it have, if you include an intercept? Build the Design matrix and solve the system using this function.


---
# Optional: Adding regularization for an underdetermined system

Regularization means adding a penalty term to the cost (or objective) function to stabilize the solution or prevent overfitting.

Why we add regularization

1. Prevent overfitting
- Without regularization, the model may “memorize” noise or outliers.
- Regularization smooths or shrinks the solution, improving generalization.

2. Stabilize ill-posed problems
- Inverse or linear systems (common in geophysics, hydrogeodesy, etc.) can be ill-conditioned or underdetermined.
- Regularization makes the system numerically stable and ensures a unique, robust solution.

3. Enforce physical or prior constraints
- In many scientific applications, we expect smoothness, continuity, or small gradients.
- Regularization can enforce these properties even when data are noisy or incomplete.

4. Control parameter size or roughness
- Penalizing large coefficients prevents extreme, physically implausible solutions.

In [ ]:

# For an  L_2-norm with added smoothing penalty for parameter using L1 norm
def regularized_norm(p0,A,l,lambd):
    #return np.linalg.norm(A@p0-l,ord=2) + lam*lam*np.linalg.norm(p0,ord=1)  # regularization for smooth parameter
    return np.sum(np.power(np.abs(A@p0 - l),2)) + lambd*lambd * np.linalg.norm(p0,ord=1)  # regularization for smooth parameter

lam = np.array([4.0, 8.0,10.0,15.0,  20.0, 25.0, 30.0, 40.0, 50.0]) # regularization parameter


In [ ]:
# convert masked arrays into plain arrays, removing noData values
OceanMask = tws.mask | soilmS.mask
x = soilmS.data[~OceanMask]  # indipendent variable
y = tws.data[~OceanMask] # dependent variable

A = np.column_stack((x, np.ones_like(x)))  # fits whatever is defined by the design matrix A
# note: intercept has to be added in the Design matrix

solr = scipy.optimize.minimize(regularized_norm,p0,args=(A,y,20)) 
p=solr.x
R2r  = r2_score(y, A@p)
R2r

In [ ]:

p0 = np.array([0,0])
i=0
resNorm = np.empty(len(lam))
lamNorm = np.empty(len(lam))
R2r = np.empty(len(lam))
for j in lam:
    solr = scipy.optimize.minimize(regularized_norm,p0,args=(A,y,j)) # linear regression with L2-Norm plus regularization
    p = solr.x
#    resNorm[i] = np.linalg.norm(A@p-y,ord=2)
#    lamNorm[i] = j*j*np.linalg.norm(p,ord=2)
    resNorm[i] = np.sum(np.power(np.abs(A@p - y),2)) 
    lamNorm[i] = j*j * np.linalg.norm(p,ord=1)
    R2r[i]  = r2_score(y, A@p)
    print(j, p, resNorm[i], lamNorm[i], R2r[i])
    i= i+1


In [ ]:
plt.figure()
plt.plot(lam,R2r,'--',color='k',linewidth=1) # L1
plt.xlabel('Lambda'), plt.ylabel('R2')
plt.show()